# Kafka -> Smart Filtering -> Delta (Databricks)

Notebook nay doc stream tu Kafka, ap dung bo loc luu tru thong minh, va ghi vao Delta table.

## Smart filtering rules
- Threshold: chi luu khi gia tri moi thay doi dang ke so voi gia tri da luu gan nhat.
- Max Interval: neu qua lau du lieu khong doi, van luu 1 ban ghi de giu trace.

Mapping hien tai: 5 sensor, moi sensor do 1 metric:
- sensor_1 -> temperature
- sensor_2 -> humidity
- sensor_3 -> soil_moisture
- sensor_4 -> light_intensity
- sensor_5 -> pressure

In [0]:
# ===== Runtime configuration =====
KAFKA_BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
KAFKA_TOPIC = "metrics"
STARTING_OFFSETS = "earliest"

KAFKA_SECURITY_PROTOCOL = "SASL_SSL"
KAFKA_SASL_MECHANISM = "PLAIN"
KAFKA_SASL_USERNAME = "SPEASAZGDLX7IJA2"
KAFKA_SASL_PASSWORD = "cfltKpuSbFUqt96XjbSQdP6ecql06q+h1qwsdAc6ZGgeH1sptHC/y6MIhvbi7Gbw"

KAFKA_JAAS_CONFIG = f"""
kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
username="{KAFKA_SASL_USERNAME}"
password="{KAFKA_SASL_PASSWORD}";
""".strip()

# Delta database/tables
DB_NAME = "iot_analytics"
TARGET_TABLE = f"{DB_NAME}.smart_filtered_measurements"
STATE_TABLE = f"{DB_NAME}.smart_filter_state"

# Checkpoint location (DBFS or UC volume)
CHECKPOINT_PATH = "/Volumes/workspace/metrics_app_streaming/checkpoints/iot_smart_filtering"

# If True, remove old checkpoint to re-read from STARTING_OFFSETS
RESET_CHECKPOINT = False

# If True, clear state table so next run stores as first_record
RESET_STATE_TABLE = False

# Rule: max waiting time before forcing a record
MAX_INTERVAL_MINUTES = 15

# If True, records older than the saved state are ignored to prevent state rollback.
SKIP_LATE_EVENTS = True

# Databricks cluster nay khong ho tro infinite ProcessingTime trigger.
# Chon 1 trong 2 gia tri: 'availableNow' hoac 'once'
STREAM_TRIGGER = "availableNow"
POLL_INTERVAL_SECONDS = 5

# Print debug stats for each micro-batch
PRINT_BATCH_STATS = True

# Fallback threshold when metric_type has no specific config
DEFAULT_THRESHOLD = 0.5

## LocaltoNet Setup Notes
Use LocaltoNet TCP tunnel for Kafka and keep bootstrap endpoint consistent between Kafka advertised listener and Databricks notebook.

1. Start a LocaltoNet TCP tunnel to local Kafka port 9092.
2. Use this LocaltoNet endpoint: utput-gross.gl.at.ply.gg:49340.
3. Restart Kafka with external advertised host/port set to that endpoint.
4. Set `LOCALTONET_BOOTSTRAP_SERVERS` in Cell 2 to `utput-gross.gl.at.ply.gg:49340`.
5. For first run after endpoint change, set `RESET_CHECKPOINT = True`.

PowerShell example on local machine:
```powershell
$env:KAFKA_EXTERNAL_HOST = "utput-gross.gl.at.ply.gg"
$env:KAFKA_EXTERNAL_PORT = "49340"
docker-compose up -d kafka
```

In [0]:
from delta.tables import DeltaTable
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions", "200")

threshold_config = [
    ("temperature", 0.50),
    ("humidity", 2.00),
    ("soil_moisture", 3.00),
    ("light_intensity", 100.0),
    ("pressure", 1.00),
]
threshold_df = spark.createDataFrame(threshold_config, ["metric_type", "threshold"])

payload_schema = T.StructType([
    T.StructField("timestamp", T.StringType()),
    T.StructField("sensor_id", T.StringType()),
    T.StructField("location", T.StringType()),
    T.StructField("metric_type", T.StringType()),
    T.StructField("unit", T.StringType()),
    T.StructField("temperature", T.DoubleType()),
    T.StructField("humidity", T.DoubleType()),
    T.StructField("soil_moisture", T.DoubleType()),
    T.StructField("light_intensity", T.DoubleType()),
    T.StructField("pressure", T.DoubleType()),
])


In [0]:
# ===== Read Kafka stream and normalize payload =====

kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .option("failOnDataLoss", "false")
    .option("kafka.security.protocol", KAFKA_SECURITY_PROTOCOL)
    .option("kafka.sasl.mechanism", KAFKA_SASL_MECHANISM)
    .option("kafka.sasl.jaas.config", KAFKA_JAAS_CONFIG)
    .load()
)

metric_value_expr = (
    F.when(F.col("metric_type") == "temperature", F.col("temperature"))
     .when(F.col("metric_type") == "humidity", F.col("humidity"))
     .when(F.col("metric_type") == "soil_moisture", F.col("soil_moisture"))
     .when(F.col("metric_type") == "light_intensity", F.col("light_intensity"))
     .when(F.col("metric_type") == "pressure", F.col("pressure"))
)

parsed_df = (
    kafka_stream_df
    .select(
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.from_json(F.col("value").cast("string"), payload_schema).alias("payload")
    )
    .select("kafka_topic", "kafka_partition", "kafka_offset", "kafka_timestamp", "payload.*")
    .withColumn("event_ts", F.coalesce(F.to_timestamp("timestamp"), F.col("kafka_timestamp")))
    .withColumn("metric_value", metric_value_expr.cast("double"))
    .filter(F.col("sensor_id").isNotNull())
    .filter(F.col("metric_type").isNotNull())
    .filter(F.col("event_ts").isNotNull())
    .filter(F.col("metric_value").isNotNull())
    .select(
        "event_ts",
        "sensor_id",
        "location",
        "metric_type",
        "unit",
        "metric_value",
        "kafka_topic",
        "kafka_partition",
        "kafka_offset"
    )
)

In [0]:
# ===== Create Delta database and tables if needed =====
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  event_ts TIMESTAMP,
  sensor_id STRING,
  location STRING,
  metric_type STRING,
  metric_value DOUBLE,
  unit STRING,
  threshold DOUBLE,
  store_reason STRING,
  kafka_topic STRING,
  kafka_partition INT,
  kafka_offset LONG,
  processed_at TIMESTAMP
) USING DELTA
PARTITIONED BY (metric_type)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
  sensor_id STRING,
  metric_type STRING,
  last_saved_value DOUBLE,
  last_saved_at TIMESTAMP,
  unit STRING,
  location STRING,
  updated_at TIMESTAMP
) USING DELTA
""")

DataFrame[]

In [0]:
# ===== Smart filtering logic in foreachBatch =====
filtered_schema = T.StructType([
    T.StructField("event_ts", T.TimestampType()),
    T.StructField("sensor_id", T.StringType()),
    T.StructField("location", T.StringType()),
    T.StructField("metric_type", T.StringType()),
    T.StructField("metric_value", T.DoubleType()),
    T.StructField("unit", T.StringType()),
    T.StructField("threshold", T.DoubleType()),
    T.StructField("store_reason", T.StringType()),
    T.StructField("kafka_topic", T.StringType()),
    T.StructField("kafka_partition", T.IntegerType()),
    T.StructField("kafka_offset", T.LongType()),
])

def evaluate_group(pdf: pd.DataFrame) -> pd.DataFrame:
    if pdf.empty:
        return pd.DataFrame(columns=[field.name for field in filtered_schema])

    pdf = pdf.sort_values(["event_ts", "kafka_offset"], kind="mergesort").reset_index(drop=True)

    threshold_series = pdf["threshold"].dropna()
    threshold_value = float(threshold_series.iloc[0]) if not threshold_series.empty else float(DEFAULT_THRESHOLD)

    first_row = pdf.iloc[0]
    last_saved_value = first_row.get("last_saved_value", None)
    last_saved_at = pd.to_datetime(first_row.get("last_saved_at", pd.NaT), errors="coerce")

    current_saved_value = None if pd.isna(last_saved_value) else float(last_saved_value)
    current_saved_at = last_saved_at
    kept_rows = []

    for _, row in pdf.iterrows():
        event_ts = pd.to_datetime(row["event_ts"], errors="coerce")
        metric_value = row["metric_value"]

        if pd.isna(event_ts) or pd.isna(metric_value):
            continue

        if SKIP_LATE_EVENTS and pd.notna(current_saved_at) and event_ts <= current_saved_at:
            continue

        if pd.isna(current_saved_at):
            store_reason = "first_record"
        else:
            value_diff = abs(float(metric_value) - float(current_saved_value))
            minutes_since_last = max((event_ts - current_saved_at).total_seconds() / 60.0, 0.0)

            if value_diff >= threshold_value:
                store_reason = "threshold"
            elif minutes_since_last >= float(MAX_INTERVAL_MINUTES):
                store_reason = "max_interval"
            else:
                continue

        kept_rows.append({
            "event_ts": event_ts.to_pydatetime(),
            "sensor_id": row["sensor_id"],
            "location": row["location"],
            "metric_type": row["metric_type"],
            "metric_value": float(metric_value),
            "unit": row["unit"],
            "threshold": threshold_value,
            "store_reason": store_reason,
            "kafka_topic": row["kafka_topic"],
            "kafka_partition": int(row["kafka_partition"]) if pd.notna(row["kafka_partition"]) else None,
            "kafka_offset": int(row["kafka_offset"]) if pd.notna(row["kafka_offset"]) else None,
        })

        current_saved_value = float(metric_value)
        current_saved_at = event_ts

    return pd.DataFrame(kept_rows, columns=[field.name for field in filtered_schema])

def smart_filter_and_store(micro_batch_df, batch_id):
    batch_count = micro_batch_df.count()

    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] input rows = {batch_count}")

    if batch_count == 0:
        return

    state_df = spark.table(STATE_TABLE).select(
        "sensor_id",
        "metric_type",
        "last_saved_value",
        "last_saved_at"
    )

    prepared_df = (
        micro_batch_df.alias("m")
        .join(threshold_df.alias("t"), on="metric_type", how="left")
        .join(state_df.alias("s"), on=["sensor_id", "metric_type"], how="left")
        .withColumn("threshold", F.coalesce(F.col("t.threshold"), F.lit(DEFAULT_THRESHOLD)))
        .select(
            F.col("m.event_ts").alias("event_ts"),
            F.col("sensor_id"),
            F.col("m.location").alias("location"),
            F.col("metric_type"),
            F.col("m.metric_value").alias("metric_value"),
            F.col("m.unit").alias("unit"),
            F.col("m.kafka_topic").alias("kafka_topic"),
            F.col("m.kafka_partition").alias("kafka_partition"),
            F.col("m.kafka_offset").alias("kafka_offset"),
            F.col("threshold"),
            F.col("s.last_saved_value").alias("last_saved_value"),
            F.col("s.last_saved_at").alias("last_saved_at")
        )
    )

    to_store_df = (
        prepared_df
        .groupBy("sensor_id", "metric_type")
        .applyInPandas(evaluate_group, schema=filtered_schema)
        .withColumn("processed_at", F.current_timestamp())
    )

    to_store_count = to_store_df.count()

    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] rows after smart filter = {to_store_count}")

        if to_store_count > 0:
            reason_stats = {
                row["store_reason"]: row["count"]
                for row in to_store_df.groupBy("store_reason").count().collect()
            }
            print(f"[Batch {batch_id}] store reason counts = {reason_stats}")

    if to_store_count == 0:
        return

    # 1) Append records that pass smart filter
    to_store_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)

    # 2) Upsert only the latest saved state for each sensor_id + metric_type
    latest_state_window = Window.partitionBy("sensor_id", "metric_type").orderBy(
        F.col("event_ts").desc(),
        F.col("kafka_offset").desc()
    )

    state_updates_df = (
        to_store_df
        .withColumn("rn", F.row_number().over(latest_state_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .select(
            "sensor_id",
            "metric_type",
            F.col("metric_value").alias("last_saved_value"),
            F.col("event_ts").alias("last_saved_at"),
            "unit",
            "location",
            F.current_timestamp().alias("updated_at")
        )
    )

    state_delta = DeltaTable.forName(spark, STATE_TABLE)

    (
        state_delta.alias("s")
        .merge(
            state_updates_df.alias("u"),
            "s.sensor_id = u.sensor_id AND s.metric_type = u.metric_type"
        )
        .whenMatchedUpdate(set={
            "last_saved_value": "u.last_saved_value",
            "last_saved_at": "u.last_saved_at",
            "unit": "u.unit",
            "location": "u.location",
            "updated_at": "u.updated_at"
        })
        .whenNotMatchedInsert(values={
            "sensor_id": "u.sensor_id",
            "metric_type": "u.metric_type",
            "last_saved_value": "u.last_saved_value",
            "last_saved_at": "u.last_saved_at",
            "unit": "u.unit",
            "location": "u.location",
            "updated_at": "u.updated_at"
        })
        .execute()
    )

    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] write completed.")

In [0]:
def process_microbatch(batch_df, batch_id):
    (
        batch_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

In [0]:
import time

POLL_INTERVAL_SECONDS = 15

while True:
    try:
        for active_query in spark.streams.active:
            print(f"Stopping existing query: {active_query.id}")
            active_query.stop()

        print("Starting one AvailableNow run...")

        query = (
            parsed_df.writeStream
            .queryName("iot_smart_filtering_stream")
            .option("checkpointLocation", CHECKPOINT_PATH)
            .trigger(availableNow=True)
            .toTable(TARGET_TABLE)
        )

        query.awaitTermination()

        print("Run finished.")
        print("Last progress:")
        print(query.lastProgress)

    except Exception as e:
        print("Run failed:", str(e))

    print(f"Sleep {POLL_INTERVAL_SECONDS}s...")
    time.sleep(POLL_INTERVAL_SECONDS)

Starting one AvailableNow run...
Run finished.
Last progress:
{'id': '4b2e0789-611f-4c56-ad6e-bd7235537ab7', 'runId': '66d1cf13-bd9b-4f3e-9321-f828f83db3fe', 'name': 'iot_smart_filtering_stream', 'timestamp': '2026-04-16T21:08:55.872Z', 'batchId': 11, 'batchDuration': 3510, 'durationMs': {'triggerExecution': 3510, 'queryPlanning': 325, 'collectSourceMetrics': 10, 'getBatch': 0, 'commitOffsets': 296, 'addBatch': 1727, 'latestOffset': 220, 'commitBatch': 703, 'walCommit': 212}, 'eventTime': {}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[metrics]]', 'startOffset': '{"metrics":{"0":381,"1":400,"2":410,"3":418,"4":385,"5":426}}', 'endOffset': '{"metrics":{"0":382,"1":402,"2":413,"3":424,"4":385,"5":429}}', 'latestOffset': '{"metrics":{"0":382,"1":402,"2":413,"3":424,"4":385,"5":429}}', 'numInputRows': 15, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 4.273504273504273, 'metrics': {'minOffsetsBehindLatest': '0', 'maxOffsetsBehindLatest': '3', 'avgOffsetsBehin

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

### Updated logic
This revised version fixes two important issues:

- `metric_value` is now derived from `metric_type`, so the stored value always matches the declared metric.
- Smart filtering now evaluates records sequentially within each `sensor_id + metric_type` micro-batch group, so important changes in the middle of a batch are not silently dropped.

It also avoids state rollback by skipping late events when `SKIP_LATE_EVENTS = True`, and only the latest stored row per key is used to update the state table.


In [0]:
# Monitoring / validation
display(
    spark.sql(
        f"""
        SELECT metric_type, store_reason, COUNT(*) AS total_rows, MAX(event_ts) AS latest_event_ts
        FROM {TARGET_TABLE}
        GROUP BY metric_type, store_reason
        ORDER BY metric_type, store_reason
        """
    )
)

display(spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY event_ts DESC LIMIT 100"))
display(spark.sql(f"SELECT * FROM {STATE_TABLE} ORDER BY updated_at DESC LIMIT 100"))


metric_type,store_reason,total_rows,latest_event_ts
humidity,null,333,2026-04-17T04:17:06.549Z
humidity,first_record,1,2026-04-17T03:28:13.553Z
humidity,threshold,16,2026-04-17T03:48:31.832Z
light_intensity,null,333,2026-04-17T04:17:06.549Z
light_intensity,first_record,1,2026-04-17T03:28:13.553Z
light_intensity,threshold,12,2026-04-17T03:44:56.292Z
pressure,null,333,2026-04-17T04:17:06.549Z
pressure,first_record,1,2026-04-17T03:28:13.553Z
pressure,max_interval,1,2026-04-17T03:43:16.057Z
soil_moisture,null,333,2026-04-17T04:17:06.549Z


event_ts,sensor_id,location,metric_type,metric_value,unit,threshold,store_reason,kafka_topic,kafka_partition,kafka_offset,processed_at
2026-04-17T04:17:06.549Z,sensor_5,Outdoor,pressure,1012.65,hPa,null,null,metrics,4,467,null
2026-04-17T04:17:06.549Z,sensor_2,Living_Room,humidity,73.08,%,null,null,metrics,4,466,null
2026-04-17T04:17:06.549Z,sensor_1,Living_Room,temperature,20.77,°C,null,null,metrics,4,465,null
2026-04-17T04:17:06.549Z,sensor_4,Outdoor,light_intensity,0.0,lux,null,null,metrics,2,492,null
2026-04-17T04:17:06.549Z,sensor_3,Garden,soil_moisture,94.18,%,null,null,metrics,1,482,null
2026-04-17T04:17:01.534Z,sensor_2,Living_Room,humidity,72.58,%,null,null,metrics,1,481,null
2026-04-17T04:17:01.534Z,sensor_1,Living_Room,temperature,20.66,°C,null,null,metrics,0,476,null
2026-04-17T04:17:01.534Z,sensor_3,Garden,soil_moisture,94.57,%,null,null,metrics,5,496,null
2026-04-17T04:17:01.534Z,sensor_5,Outdoor,pressure,1012.53,hPa,null,null,metrics,0,477,null
2026-04-17T04:17:01.534Z,sensor_4,Outdoor,light_intensity,0.0,lux,null,null,metrics,2,491,null


sensor_id,metric_type,last_saved_value,last_saved_at,unit,location,updated_at
sensor_3,soil_moisture,81.11,2026-04-17T03:49:17.032Z,%,Garden,2026-04-16T20:49:59.760Z
sensor_4,light_intensity,7.0,2026-04-17T03:44:56.292Z,lux,Outdoor,2026-04-16T20:49:59.760Z
sensor_1,temperature,20.15,2026-04-17T03:48:51.978Z,°C,Living_Room,2026-04-16T20:49:59.760Z
sensor_2,humidity,73.17,2026-04-17T03:48:31.832Z,%,Living_Room,2026-04-16T20:49:59.760Z
sensor_5,pressure,1012.12,2026-04-17T03:43:16.057Z,hPa,Outdoor,2026-04-16T20:49:59.760Z


## Stop stream

Chay cell sau khi muon dung stream:

```python
query.stop()
```

In [0]:
for q in spark.streams.active:
    print(f"Stopping: {q.name}")
    q.stop()